# מעבדה 01 — המתנד ההרמוני הפשוט

במעבדה הזו תעמידו את האיזוכרוניות למבחן — תמדדו האם זמן המחזור באמת
מתעלם מהמשרעת — תצפו באנרגיה גולשת הלוך ושוב תחת סכום שטוח, ואז תתייחסו אל
הסימולציה כאל מכשיר רועש ותחלצו ממנה את $\omega_0$ בצירוף רף שגיאה כן.

עבדו לפי הסדר. במקום שבו המחברת מבקשת מכם לנבא, כתבו את הניבוי שלכם בתא
המיועד לכך **לפני** הרצת התא הבא.

## מפרט המודל

| | |
|---|---|
| **מערכת** | מסת נקודה $m$ על קפיץ ליניארי חסר מסה בעל קבוע קפיץ $k$, בממד אחד |
| **דינמיקה** | $m\ddot{x} = -kx - b\dot{x}$, צורות סגורות במקום שבו הן קיימות, velocity-Verlet במקומות אחרים |
| **שפה** | אין — המסה נעה על ישר אינסופי ושום דבר אינו מוחלף |
| **אנסמבל** | מסלול דטרמיניסטי יחיד עבור כל $(x_0, v_0)$; רעש המדידה גאוסי, בלתי תלוי, ובעל זרע קבוע |
| **מוזנח** | מסת הקפיץ, אי-ליניאריות בהתמתחות גדולה, וכל מנגנון מיקרוסקופי מאחורי $b$ |
| **תקף כאשר** | ההזזות נותרות בתחום הליניארי וצעד הזמן נותר נמוך משמעותית מזמן המחזור |
| **אופני כשל** | משרעות שחשות את אי-הליניאריות; צעדים קרובים לגבול היציבות; שאלה לאן הולכת האנרגיה |

כל הפיזיקה נמצאת ב-`wavelab.oscillators` וב-`wavelab.measurement` — פתחו וקראו
אותן.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Those builds still have to be asked for by name:
# the kernel fetches a Pyodide package by itself only when a cell's own text imports it, and
# scipy is imported inside wavelab, where it does not look. Add any new compiled dependency
# of wavelab to the first list below and any new *pure-Python* one to the second.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["numpy", "scipy"])
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import measurement, oscillators
from wavelab.validation import seed_study

MASS = 0.5  # kg
STIFFNESS = 8.0  # N/m
OMEGA0 = oscillators.natural_frequency(MASS, STIFFNESS)
PERIOD = 2 * np.pi / OMEGA0

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"omega0 = {OMEGA0} rad/s   period = {PERIOD:.4f} s")

## חלק 1 — התנועה

שחררו את העגלה ממנוחה ב-$x_0 = 0.1$ m. משמאל: מיקום ומהירות (בקנה מידה) —
צפו בפיגור רבע המחזור. מימין: אותה תנועה משורטטת ב*מרחב הפאזה*, המישור
$(x, v)$, שבו כל ההיסטוריה היא אליפסה סגורה אחת.

In [ ]:
t = np.linspace(0.0, 3 * PERIOD, 1200)
x = oscillators.position(t, MASS, STIFFNESS, 0.1, 0.0)
v = oscillators.velocity(t, MASS, STIFFNESS, 0.1, 0.0)

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.6))
left.plot(t, x, label="x (m)")
left.plot(t, v / OMEGA0, label="v/ω₀ (m)")
left.set_xlabel("t (s)"), left.legend()
left.set_title("velocity leads position by a quarter period")
right.plot(x, v, lw=1.2)
right.set_xlabel("x (m)"), right.set_ylabel("v (m/s)")
right.set_title("phase space: the energy ellipse")
plt.tight_layout()
plt.show()

## חלק 2 — האיזוכרוניות במבחן

### לנבא

1. שחררו מ-$0.05$ m במקום מ-$0.10$ m: האם זמן המחזור משתנה? בכמה?
2. שמרו על נקודת השחרור אך *הכפילו את המסה פי ארבעה*: מה קורה לזמן המחזור?
3. היכן העגלה נעה מהר ביותר — בנקודת השחרור, במרכז, או אי-שם
   בין לבין?

כתבו את הניבוי שלכם **לפני** הרצת התא הבא.

**הניבוי שלכם:**

In [ ]:
def fitted_period(mass, stiffness, x0):
    span = 6 * 2 * np.pi / oscillators.natural_frequency(mass, stiffness)
    tt = np.linspace(0.0, span, 2400)
    fit = measurement.fit_cosine(tt, oscillators.position(tt, mass, stiffness, x0, 0.0))
    return 2 * np.pi / fit.omega


amplitudes = [0.02, 0.05, 0.1, 0.2]
periods = [fitted_period(MASS, STIFFNESS, a) for a in amplitudes]
for a, p in zip(amplitudes, periods, strict=True):
    print(f"release amplitude {a:5.2f} m   ->   period {p:.6f} s")

heavy = fitted_period(4 * MASS, STIFFNESS, 0.1)
print(f"quadrupled mass, same release  ->   period {heavy:.6f} s   ({heavy / periods[2]:.3f}x)")

גורם עשר במשרעת אינו מזיז את זמן המחזור; גורם ארבע במסה מכפיל אותו.
שני אלה הם $T = 2\pi\sqrt{m/k}$ מדברת: המשרעת פשוט אינה בנוסחה.

## חלק 3 — חילופי האנרגיה

צפו ב-$K$ וב-$U$ מתחלפים בעוד סכומם נותר שטוח — וספרו כמה פעמים האנרגיה
הקינטית מגיעה לשיא במחזור לפני קריאת הכותרת.

In [ ]:
kinetic, potential, total = oscillators.energies(x, v, MASS, STIFFNESS)

plt.figure(figsize=(9, 3.6))
plt.plot(t, kinetic, label="K")
plt.plot(t, potential, label="U")
plt.plot(t, total, lw=2, label="K + U")
plt.xlabel("t (s)"), plt.ylabel("energy (J)"), plt.legend()
plt.title("K peaks twice per cycle — once per centre crossing — under a flat total")
plt.tight_layout()
plt.show()
print(f"relative variation of the total: {(total.max() - total.min()) / total[0]:.2e}")

## חלק 4 — סימולציה כניסוי

הצורה הסגורה היא מותרות של המתנד ה*ליניארי*; כמעט כל דבר בהמשך הקורס חייב
להיות משולב נומרית. לכן: בדקו את המשלב מול הצורה הסגורה — ואז קלקלו את
האות ברעש גלאי ו*מדדו* את $\omega_0$ בחזרה ממנו, בצירוף רף שגיאה.

In [ ]:
dt = oscillators.max_stable_dt(MASS, STIFFNESS)
trajectory = oscillators.simulate(MASS, STIFFNESS, 0.1, 0.0, dt, 3000)
exact = oscillators.position(trajectory.times, MASS, STIFFNESS, 0.1, 0.0)
worst = np.max(np.abs(trajectory.positions - exact))
print(f"max |numerical - exact| over 30 periods: {worst:.2e} m")

# The virtual apparatus: the exact signal, corrupted by detector noise, then fitted.
noisy = measurement.add_noise(exact, 0.005, rng)
fit = measurement.fit_cosine(trajectory.times, noisy)

plt.figure(figsize=(9, 3.6))
plt.plot(trajectory.times, noisy, ".", ms=2, alpha=0.4, label="measured (noisy)")
plt.plot(trajectory.times,
         fit.amplitude * np.cos(fit.omega * trajectory.times + fit.phase) + fit.offset,
         lw=1.5, label="cosine fit")
plt.xlabel("t (s)"), plt.ylabel("x (m)"), plt.legend()
plt.tight_layout()
plt.show()
print(f"omega_fit = {fit.omega:.4f} ± {fit.omega_err:.4f} rad/s   vs   sqrt(k/m) = {OMEGA0:.4f}")

## חלק 5 — בדיקות אוטומטיות

הטענות שהמחברת הזו העלתה, כאסרציות הניתנות להרצה — אותן בדיקות בדיוק
רצות בחבילת הבדיקות של הפרויקט, כך שהעמוד, המחברת והספרייה אינם יכולים
לסטות זה מזה בשקט.

In [ ]:
# 1. Integrator vs closed form at a fine step: better than one part in 10^3 of amplitude.
fine = oscillators.simulate(MASS, STIFFNESS, 0.1, 0.2, PERIOD / 1000, 20_000)
reference = oscillators.position(fine.times, MASS, STIFFNESS, 0.1, 0.2)
scale, _ = oscillators.amplitude_phase(0.1, 0.2, OMEGA0)
assert np.max(np.abs(fine.positions - reference)) / scale < 1e-3

# 2. Energy over 100 periods: bounded oscillation at (omega0 dt)^2 / 4, and no drift.
steps = round(PERIOD / dt)
check = oscillators.simulate(MASS, STIFFNESS, 0.1, 0.0, dt, 100 * steps)
_, _, total_check = oscillators.energies(check.positions, check.velocities, MASS, STIFFNESS)
bound = (OMEGA0 * dt) ** 2 / 4
assert (total_check.max() - total_check.min()) / total_check[0] < 1.5 * bound
drift = abs(total_check[-steps:].mean() - total_check[:steps].mean()) / total_check[0]
assert drift < 1e-7

# 3. The measurement pipeline recovers omega0 within its scatter, across 8 seeds.
tt = np.linspace(0.0, 8 * PERIOD, 1601)
clean = oscillators.position(tt, MASS, STIFFNESS, 0.1, 0.0)
study = seed_study(
    lambda r: measurement.fit_cosine(tt, measurement.add_noise(clean, 0.02, r), OMEGA0).omega,
    n_seeds=8,
)
assert study.agrees_with(OMEGA0, n_sigma=3.0)

print("all checks passed")
wobble = (total_check.max() - total_check.min()) / total_check[0]
print(f"energy wobble {wobble:.2e}   (bound {bound:.2e})")
print(f"omega measured {study.mean:.4f} ± {study.standard_error:.1e}   vs   {OMEGA0}")

## חלק 6 — לחקור בעצמכם

המשלב רץ כאן באופן חי, ולכן הפאנל הזה משתמש בכפתור **הרצה** (`interact_manual`):
האינטגרציה איטית מדי בליבת הדפדפן כדי לרוץ מחדש בכל תזוזה של מחוון.

שני ניסויים ששווה לבצע:

1. תנו לעגלה מהירות התחלתית בנוסף להזזה, וצפו במה שמשתנה באליפסת מרחב
   הפאזה — ובמה ש*אינו* משתנה בזמן המחזור.
2. הגבירו את הריסון מאפס (תצוגה מקדימה של מודול 02) וצפו באליפסה הופכת
   לספירלה: אנרגיה עוזבת, זמן המחזור כמעט אינו זז.

In [ ]:
import ipywidgets as widgets


def explore(mass=0.5, stiffness=8.0, x0=0.1, v0=0.0, damping=0.0):
    step = oscillators.max_stable_dt(mass, stiffness)
    run = oscillators.simulate(mass, stiffness, x0, v0, step, 4000, damping=damping)
    _, _, total_e = oscillators.energies(run.positions, run.velocities, mass, stiffness)

    fig, (left, middle, right) = plt.subplots(1, 3, figsize=(12, 3.4))
    left.plot(run.times, run.positions, lw=1.0)
    left.set_xlabel("t (s)"), left.set_ylabel("x (m)"), left.set_title("position")
    middle.plot(run.positions, run.velocities, lw=0.8)
    middle.set_xlabel("x (m)"), middle.set_ylabel("v (m/s)"), middle.set_title("phase space")
    right.plot(run.times, total_e, lw=1.2)
    right.set_xlabel("t (s)"), right.set_ylabel("E (J)"), right.set_title("total energy")
    plt.tight_layout()
    plt.show()
    print(f"omega0 = {oscillators.natural_frequency(mass, stiffness):.3f} rad/s")


widgets.interact_manual(
    explore,
    mass=widgets.FloatSlider(min=0.1, max=4.0, step=0.1, value=0.5, description="m (kg)"),
    stiffness=widgets.FloatSlider(min=1.0, max=32.0, step=0.5, value=8.0, description="k (N/m)"),
    x0=widgets.FloatSlider(min=-0.2, max=0.2, step=0.01, value=0.1, description="x0 (m)"),
    v0=widgets.FloatSlider(min=-0.5, max=0.5, step=0.05, value=0.0, description="v0 (m/s)"),
    damping=widgets.FloatSlider(min=0.0, max=2.0, step=0.05, value=0.0, description="b (kg/s)"),
);

## בדקו את הבנתכם

הריצו את התא שלהלן לחידון עם בדיקה אוטומטית. אותן שאלות, בתוספת הסברים
כתובים לכל אפשרות, נמצאות בעמוד המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "01-sho.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בהיגיון שלכם?
2. הקפיץ של המודל ליניארי לחלוטין. נקבו בתצפית אחת מהיום שהייתה משתנה
   אילו זה לא היה כך — ואחת ששרדה.
3. הסבירו, בלי משוואות, מדוע זמן המחזור מתעלם מהמשרעת עבור חוק הכוח הזה
   — ומדוע הטיעון הזה נכשל עבור מטוטלת בזוויות גדולות.

**התשובות שלכם:**

1.
2.
3.